# VoiceShield (BAFV-PCTA) — Phase 7 Training, Calibration & Evaluation

Run this notebook on **Google Colab (Runtime -> Change runtime type -> T4 GPU)** to:
1. Load a public speech deepfake dataset (`mueller91/In-The-Wild` or `Bisher/ASVspoof_2019_LA` from Hugging Face, or your Google Drive).
2. Train real anti-spoofing weights (`models/aasist_baseline.pt`) with 8 kHz telephony augmentation.
3. Calibrate empirical BAFV & Open-Set Generator centroids (`models/calibration_stats.json`).
4. Run the Phase 7 benchmark evaluation (`EER`, `min-tDCF`, `ROC-AUC`, Telephony Guard ablation, and LOGO Open-Set).
5. Download the trained artifacts ready to push to GitHub (`models/`).

In [ ]:
#@title 1. Check GPU, Clone VoiceShield & Install Dependencies
!nvidia-smi
%cd /content
!rm -rf Capstone-Project
!git clone https://github.com/Rahul1062005/Capstone-Project.git
%cd /content/Capstone-Project
!pip install -q -r requirements.txt datasets
import torch
print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())

## 2. Load a Real Speech Deepfake Dataset
Use **Cell 2A** for zero-setup 1-click streaming from Hugging Face (`mueller91/In-The-Wild`), or **Cell 2B** if you stored ASVspoof 2019 LA / custom folders in Google Drive.

In [ ]:
#@title 2A. Option A (Zero-Setup) — Stream Real & Fake Clips from Hugging Face In-The-Wild
import os
from pathlib import Path
import numpy as np
import soundfile as sf
from datasets import load_dataset

DATASET_MODE = 'folder'
DATA_DIR = '/content/dataset'
MAX_PER_CLASS = 400 #@param {type:'integer'}

bonafide_dir = Path(DATA_DIR) / 'bonafide'
spoof_dir = Path(DATA_DIR) / 'spoof'
bonafide_dir.mkdir(parents=True, exist_ok=True)
spoof_dir.mkdir(parents=True, exist_ok=True)

ds_raw = load_dataset('mueller91/In-The-Wild', split='train', streaming=True)
label_feat = ds_raw.features.get('label') if ds_raw.features else None
print('Dataset label feature:', label_feat)

ds = ds_raw.shuffle(seed=42, buffer_size=400)
counts = {0: 0, 1: 0}

for idx, row in enumerate(ds):
    raw_lbl = row.get('label', '')
    if hasattr(label_feat, 'int2str') and isinstance(raw_lbl, int):
        label_str = label_feat.int2str(raw_lbl).lower()
    else:
        label_str = str(raw_lbl).lower()

    is_bonafide = label_str in ('bona-fide', 'bonafide', 'real', 'human', 'genuine', '0')
    cls_id = 0 if is_bonafide else 1
    if counts[cls_id] >= MAX_PER_CLASS:
        if counts[0] >= MAX_PER_CLASS and counts[1] >= MAX_PER_CLASS:
            break
        continue

    audio = row['audio']
    arr = np.asarray(audio['array'], dtype=np.float32)
    sr = int(audio['sampling_rate'])
    if len(arr) < int(0.6 * sr):
        continue

    out_dir = bonafide_dir if cls_id == 0 else spoof_dir
    sf.write(out_dir / f'itw_{cls_id}_{counts[cls_id]:04d}.wav', arr, sr)
    counts[cls_id] += 1
    total_saved = counts[0] + counts[1]
    if total_saved % 100 == 0:
        print(f'Progress: bonafide={counts[0]}/{MAX_PER_CLASS}, spoof={counts[1]}/{MAX_PER_CLASS}')

print(f'Done! Exported bonafide={counts[0]}, spoof={counts[1]} into {DATA_DIR}')

In [ ]:
#@title 2B. Option B (Optional) — Mount Google Drive for ASVspoof 2019 LA or Custom Data
USE_DRIVE = False #@param {type:'boolean'}
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DATASET_MODE = 'folder'  # change to 'asvspoof' for protocol files
    DATA_DIR = '/content/drive/MyDrive/voiceshield_dataset'
    ASV_TRAIN_PROTO = '/content/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt'
    ASV_TRAIN_AUDIO = '/content/LA/ASVspoof2019_LA_train/flac'
    ASV_DEV_PROTO = '/content/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.dev.trl.txt'
    ASV_DEV_AUDIO = '/content/LA/ASVspoof2019_LA_dev/flac'

In [ ]:
#@title 3. Train Baseline Anti-Spoofing Detector (saves models/aasist_baseline.pt)
EPOCHS = 20 #@param {type:'integer'}
BATCH_SIZE = 32 #@param {type:'integer'}
TELEPHONY_AUG_PROB = 0.25 #@param {type:'number'}

if DATASET_MODE == 'folder':
    !python scripts/train_detector.py \
        --data-dir "$DATA_DIR" \
        --epochs $EPOCHS \
        --batch-size $BATCH_SIZE \
        --telephony-aug-prob $TELEPHONY_AUG_PROB \
        --output-checkpoint models/aasist_baseline.pt \
        --metrics-out models/training_metrics.json
else:
    !python scripts/train_detector.py \
        --asvspoof-protocol "$ASV_TRAIN_PROTO" \
        --audio-root "$ASV_TRAIN_AUDIO" \
        --dev-protocol "$ASV_DEV_PROTO" \
        --dev-audio-root "$ASV_DEV_AUDIO" \
        --epochs $EPOCHS \
        --batch-size $BATCH_SIZE \
        --telephony-aug-prob $TELEPHONY_AUG_PROB \
        --output-checkpoint models/aasist_baseline.pt \
        --metrics-out models/training_metrics.json

In [ ]:
#@title 4. Calibrate BAFV Coupling & Open-Set Generator Centroids
BONAFIDE_DIR = f"{DATA_DIR}/bonafide"
SPOOF_BY_GEN_DIR = ''  # Optional: path with subfolders per generator (e.g. ElevenLabs/, RVC/, Bark/)

if SPOOF_BY_GEN_DIR:
    !python scripts/calibrate_manifold.py \
        --bonafide-dir "$BONAFIDE_DIR" \
        --spoof-root "$SPOOF_BY_GEN_DIR" \
        --checkpoint models/aasist_baseline.pt \
        --output models/calibration_stats.json \
        --max-files 500
else:
    !python scripts/calibrate_manifold.py \
        --bonafide-dir "$BONAFIDE_DIR" \
        --checkpoint models/aasist_baseline.pt \
        --output models/calibration_stats.json \
        --max-files 500

In [ ]:
#@title 5. Run Phase 7 Evaluation (Clean vs. 8 kHz Telephony + LOGO + min-tDCF)
MAX_EVAL_SAMPLES = 1000 #@param {type:'integer'}

if DATASET_MODE == 'folder':
    extra = f'--spoof-root "{SPOOF_BY_GEN_DIR}"' if SPOOF_BY_GEN_DIR else ''
    !python scripts/evaluate_phase7.py \
        --data-dir "$DATA_DIR" \
        --checkpoint models/aasist_baseline.pt \
        --calibration models/calibration_stats.json \
        --max-samples $MAX_EVAL_SAMPLES \
        $extra
else:
    !python scripts/evaluate_phase7.py \
        --asvspoof-protocol "$ASV_DEV_PROTO" \
        --audio-root "$ASV_DEV_AUDIO" \
        --checkpoint models/aasist_baseline.pt \
        --calibration models/calibration_stats.json \
        --max-samples $MAX_EVAL_SAMPLES

!cat docs/PHASE_7_EVALUATION_REPORT.md

In [ ]:
#@title 6. Download Trained Weights & Phase 7 Report
from google.colab import files

for artifact in [
    'models/aasist_baseline.pt',
    'models/training_metrics.json',
    'models/calibration_stats.json',
    'docs/PHASE_7_EVALUATION_REPORT.md',
    'data/predictions/phase7_eval.json',
]:
    try:
        files.download(artifact)
    except Exception as exc:
        print('Skipped:', artifact, exc)